#### Import lib

In [1]:
import pandas as pd

#### Read file and convert parquet to dataframe

In [94]:
file = '../../data/2.bronze/ingest_bronze.parquet'

df_file = pd.read_parquet(file)

df = pd.DataFrame(df_file)

df

,id,issue_type,status,priority,assignee,timestamps
0,JIRA-0001,Bug,Open,Low,"[{'email': 'gionni.lucio@fasttrack.com', 'id':...","[{'created_at': '2025-08-02T14:55:05Z', 'resol..."
1,JIRA-0002,Bug,Resolved,High,"[{'email': 'francinne@fasttrack.com', 'id': 'u...","[{'created_at': '2025-03-09T12:39:26Z', 'resol..."
2,JIRA-0003,Story,Open,High,"[{'email': 'adyna@fasttrack.com', 'id': 'u005'...","[{'created_at': '2025-06-30T17:06:48Z', 'resol..."
3,JIRA-0004,Story,Done,High,"[{'email': 'francinne@fasttrack.com', 'id': 'u...","[{'created_at': '2025-11-22T20:24:58Z', 'resol..."
4,JIRA-0005,Task,Resolved,Medium,"[{'email': 'francinne@fasttrack.com', 'id': 'u...","[{'created_at': '2025-01-06T01:08:44Z', 'resol..."
...,...,...,...,...,...,...
995,JIRA-0996,Task,Done,Low,"[{'email': 'francinne@fasttrack.com', 'id': 'u...","[{'created_at': '2026-02-30T25:61:00Z', 'resol..."
996,JIRA-0997,Task,Open,Medium,"[{'email': 'guilherme@fasttrack.com', 'id': 'u...","[{'created_at': '2026-02-30T25:61:00Z', 'resol..."
997,JIRA-0998,Task,Resolved,Low,"[{'email': 'matheus.malta@fasttrack.com', 'id'...","[{'created_at': '2026-02-30T25:61:00Z', 'resol..."
998,JIRA-0999,Task,Done,High,"[{'email': 'adyna@fasttrack.com', 'id': 'u005'...","[{'created_at': '2026-02-30T25:61:00Z', 'resol..."


#### Copy dataframe, explode columns "assignee" and "timestamps" and expand the dictionaries into columns using "json_normalize" function

In [108]:
df_step = df.copy()

df_step = df_step.explode('assignee')
df_step = df_step.explode('timestamps')

assignee_colunms = pd.json_normalize(df_step['assignee'])
timestamps_colunms = pd.json_normalize(df_step['timestamps'])

#### Drop old columns and concatenate new columns

In [110]:
df = pd.concat(
    [
        df_step.drop(columns=['assignee', 'timestamps']),
        assignee_colunms,
        timestamps_colunms
    ],
    axis=1
)

df

,id,issue_type,status,priority,email,id,name,created_at,resolved_at
0,JIRA-0001,Bug,Open,Low,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-08-02T14:55:05Z,NaN
1,JIRA-0002,Bug,Resolved,High,francinne@fasttrack.com,u004,Francinne,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
2,JIRA-0003,Story,Open,High,adyna@fasttrack.com,u005,Adyna,2025-06-30T17:06:48Z,NaN
3,JIRA-0004,Story,Done,High,francinne@fasttrack.com,u004,Francinne,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
4,JIRA-0005,Task,Resolved,Medium,francinne@fasttrack.com,u004,Francinne,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
...,...,...,...,...,...,...,...,...,...
995,JIRA-0996,Task,Done,Low,francinne@fasttrack.com,u004,Francinne,2026-02-30T25:61:00Z,not_a_date
996,JIRA-0997,Task,Open,Medium,guilherme@fasttrack.com,u003,Guilherme Francisco,2026-02-30T25:61:00Z,not_a_date
997,JIRA-0998,Task,Resolved,Low,matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-30T25:61:00Z,not_a_date
998,JIRA-0999,Task,Done,High,adyna@fasttrack.com,u005,Adyna,2026-02-30T25:61:00Z,not_a_date


#### Handling non-standard records to enable string conversion to date and time

In [114]:
df['created_at'] = df['created_at'].replace('2026-02-30T25:61:00Z','2026-02-28T23:59:59Z')
df['resolved_at'] = df['resolved_at'].replace('not_a_date',None)

# df['created_at'].value_counts()

df

,id,issue_type,status,priority,email,id,name,created_at,resolved_at
0,JIRA-0001,Bug,Open,Low,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-08-02T14:55:05Z,NaN
1,JIRA-0002,Bug,Resolved,High,francinne@fasttrack.com,u004,Francinne,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
2,JIRA-0003,Story,Open,High,adyna@fasttrack.com,u005,Adyna,2025-06-30T17:06:48Z,NaN
3,JIRA-0004,Story,Done,High,francinne@fasttrack.com,u004,Francinne,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
4,JIRA-0005,Task,Resolved,Medium,francinne@fasttrack.com,u004,Francinne,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
...,...,...,...,...,...,...,...,...,...
995,JIRA-0996,Task,Done,Low,francinne@fasttrack.com,u004,Francinne,2026-02-28T23:59:59Z,NaN
996,JIRA-0997,Task,Open,Medium,guilherme@fasttrack.com,u003,Guilherme Francisco,2026-02-28T23:59:59Z,NaN
997,JIRA-0998,Task,Resolved,Low,matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-28T23:59:59Z,NaN
998,JIRA-0999,Task,Done,High,adyna@fasttrack.com,u005,Adyna,2026-02-28T23:59:59Z,NaN


#### Convert string to UTC date and time

In [116]:
df['created_at'] = pd.to_datetime(df['created_at'], utc=True)
df['resolved_at'] = pd.to_datetime(df['resolved_at'], utc=True)

df

,id,issue_type,status,priority,email,id,name,created_at,resolved_at
0,JIRA-0001,Bug,Open,Low,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-08-02 14:55:05+00:00,NaT
1,JIRA-0002,Bug,Resolved,High,francinne@fasttrack.com,u004,Francinne,2025-03-09 12:39:26+00:00,2025-03-10 10:39:26+00:00
2,JIRA-0003,Story,Open,High,adyna@fasttrack.com,u005,Adyna,2025-06-30 17:06:48+00:00,NaT
3,JIRA-0004,Story,Done,High,francinne@fasttrack.com,u004,Francinne,2025-11-22 20:24:58+00:00,2025-11-23 07:24:58+00:00
4,JIRA-0005,Task,Resolved,Medium,francinne@fasttrack.com,u004,Francinne,2025-01-06 01:08:44+00:00,2025-01-07 09:08:44+00:00
...,...,...,...,...,...,...,...,...,...
995,JIRA-0996,Task,Done,Low,francinne@fasttrack.com,u004,Francinne,2026-02-28 23:59:59+00:00,NaT
996,JIRA-0997,Task,Open,Medium,guilherme@fasttrack.com,u003,Guilherme Francisco,2026-02-28 23:59:59+00:00,NaT
997,JIRA-0998,Task,Resolved,Low,matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-28 23:59:59+00:00,NaT
998,JIRA-0999,Task,Done,High,adyna@fasttrack.com,u005,Adyna,2026-02-28 23:59:59+00:00,NaT


#### Rename columns

In [118]:
df.columns = ['issue_id','issue_type','issue_status','issue_priority','assignee_email','assignee_id','assignee_name','issue_created_at','issue_resolved_at']

# df.head(5)

df

,issue_id,issue_type,issue_status,issue_priority,assignee_email,assignee_id,assignee_name,issue_created_at,issue_resolved_at
0,JIRA-0001,Bug,Open,Low,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-08-02 14:55:05+00:00,NaT
1,JIRA-0002,Bug,Resolved,High,francinne@fasttrack.com,u004,Francinne,2025-03-09 12:39:26+00:00,2025-03-10 10:39:26+00:00
2,JIRA-0003,Story,Open,High,adyna@fasttrack.com,u005,Adyna,2025-06-30 17:06:48+00:00,NaT
3,JIRA-0004,Story,Done,High,francinne@fasttrack.com,u004,Francinne,2025-11-22 20:24:58+00:00,2025-11-23 07:24:58+00:00
4,JIRA-0005,Task,Resolved,Medium,francinne@fasttrack.com,u004,Francinne,2025-01-06 01:08:44+00:00,2025-01-07 09:08:44+00:00
...,...,...,...,...,...,...,...,...,...
995,JIRA-0996,Task,Done,Low,francinne@fasttrack.com,u004,Francinne,2026-02-28 23:59:59+00:00,NaT
996,JIRA-0997,Task,Open,Medium,guilherme@fasttrack.com,u003,Guilherme Francisco,2026-02-28 23:59:59+00:00,NaT
997,JIRA-0998,Task,Resolved,Low,matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-28 23:59:59+00:00,NaT
998,JIRA-0999,Task,Done,High,adyna@fasttrack.com,u005,Adyna,2026-02-28 23:59:59+00:00,NaT


#### Convert dataframe to parquet and save into 3.silver fold

In [119]:
df.to_parquet("../../data/3.silver/transform_silver.parquet", index=False)